# nanochat, as a short MPS smoke

This notebook walks the [karpathy/nanochat](https://github.com/karpathy/nanochat) pipeline in the order `runs/runcpu.sh` runs it. The code lives in `../upstream`. Nothing here downloads data or defines a second GPT. After `smoke_mps.sh` has run, the code cell below reads `../cache/smoke.log`.

## Dataset

`nanochat.dataset` downloads ClimbMix parquet shards into `NANOCHAT_BASE_DIR/base_data_climbmix`. The smoke asks for two train shards. The module always adds the pinned validation shard as well. Later stages iterate those parquets; they do not ship their own corpus.

## BPE tokenizer

`scripts.tok_train` trains a rustbpe tokenizer (`nanochat.tokenizer.RustBPETokenizer`) and writes it to `cache/tokenizer`. The full Mac demo uses 2B characters and vocab 32768. The smoke uses 200k characters and vocab 512, which still leaves room for the 256 byte tokens plus the nine special tokens (`<|bos|>`, the user/assistant markers, and the python tool markers). `scripts.tok_eval` then prints compression on a few fixed strings. It does not train.

## Pretrain

`scripts.base_train` is next-token training. `--depth` is the size dial: width is `depth * aspect_ratio`, rounded up to a multiple of `--head-dim`. The smoke uses `--depth=4`, `--head-dim=64`, `--max-seq-len=512`, `--device-batch-size=1`, `--total-batch-size=512`, and `--num-iterations=20`. `--window-pattern=L` is full attention, which is what PyTorch SDPA can run. `--core-metric-every=-1` skips the CORE eval. The script prints a line `loss:` on every step.

## SFT

`scripts.chat_sft` loads the base checkpoint and fine-tunes on conversations. The full script mixes SmolTalk, MMLU, and GSM8K. The smoke sets `--mmlu-epochs=0` and `--gsm8k-epochs=0`, keeps `--num-iterations=5`, and reuses the tiny batch. The loss lines use the same `loss:` format as pretrain.

## Chat CLI

`scripts.chat_cli` loads the SFT checkpoint and answers one prompt. The smoke prompt is `What is the capital of France?`. Twenty pretrain steps and five SFT steps are not enough to answer it. The reply is there so the pipeline reaches generation.

In [1]:
from pathlib import Path

log_path = Path("../cache/smoke.log")
settings = {
    "device": "mps",
    "shards": 2,
    "tok_max_chars": 200_000,
    "vocab_size": 512,
    "depth": 4,
    "max_seq_len": 512,
    "device_batch_size": 1,
    "pretrain_steps": 20,
    "sft_steps": 5,
    "prompt": "What is the capital of France?",
}
print("smoke settings")
for key, value in settings.items():
    print(f"  {key}: {value}")

if not log_path.exists():
    print("\nno smoke log yet. run smoke_mps.sh from the experiment folder.")
else:
    lines = log_path.read_text(errors="replace").splitlines()
    loss_lines = [line for line in lines if "| loss:" in line]
    print("\nlast loss lines")
    for line in loss_lines[-4:]:
        print(line)
    reply = ""
    for line in lines:
        if "Assistant:" in line:
            reply = line.split("Assistant:", 1)[1].strip()
    elapsed = next((line for line in lines if line.startswith("smoke elapsed_s:")), "")
    print("\nchat reply")
    print(reply or "(reply not found in the log)")
    if elapsed:
        print(elapsed)


smoke settings
  device: mps
  shards: 2
  tok_max_chars: 200000
  vocab_size: 512
  depth: 4
  max_seq_len: 512
  device_batch_size: 1
  pretrain_steps: 20
  sft_steps: 5
  prompt: What is the capital of France?

last loss lines
step 00001 (40.00%) | loss: 3.287051 | lrm: 1.00 | dt: 422.15ms | tok/sec: 1,212 | mfu: 0.00 | epoch: 1 | total time: 0.00m
step 00002 (60.00%) | loss: 4.382060 | lrm: 0.80 | dt: 45.46ms | tok/sec: 11,263 | mfu: 0.00 | epoch: 1 | total time: 0.00m
step 00003 (80.00%) | loss: 4.929932 | lrm: 0.40 | dt: 40.35ms | tok/sec: 12,688 | mfu: 0.00 | epoch: 1 | total time: 0.00m
step 00004 (100.00%) | loss: 5.146000 | lrm: 0.00 | dt: 40.31ms | tok/sec: 12,701 | mfu: 0.00 | epoch: 1 | total time: 0.00m

chat reply
oxc._j for':ingur  these otherrecur`n re:ifindcomedyn' m m forur_indt opth`d<|assistant_end|>
smoke elapsed_s: 17


## What the log is saying

The settings cell restates the smoke knobs. The last four `loss:` lines are the SFT steps. On this run they go from 3.287 to 5.146 while the learning-rate multiplier falls from 1 to 0. That is a handful of updates on conversations that happened to fit in 512 tokens. The assistant line is whatever `chat_cli` decoded. The nonsense reply is the expected outcome of a few dozen steps.

Pretrain, earlier in the same log, ends at loss 6.287 and validation bpb 4.243. The checkpoints are `cache/base_checkpoints/d4/model_000020.pt` and `cache/chatsft_checkpoints/d4/model_000004.pt`.